# Implementasi Fine-Tuning Qwen 2.5 dengan Unsloth (QLoRA)

# environment setup

In [ ]:
import torch
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"GPU: {gpu_stats.name} ({max_memory} GB)")
print(f"Initial VRAM Reserved: {start_gpu_memory} GB")

In [ ]:
# Sel 3: Inisialisasi BitsAndBytesConfig, Model & Tokenizer
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

model_id = "Qwen/Qwen2.5-3B-Instruct"


# Konfigurasi 4-bit NF4 Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True,
    padding_side="right"
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load Base Model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Persiapkan model untuk k-bit training (freeze base weights + enable gradient checkpointing)
model = prepare_model_for_kbit_training(model)
print(" Base Model berhasil dimuat dalam 4-bit NF4!")